# 1. Creating table

In [0]:
%sql
CREATE OR REPLACE TABLE cpt_utility_catalog.gold.dim_date
CLUSTER BY (date_key)
AS
WITH date_seq AS (
  SELECT explode(sequence(DATE '1999-01-01', DATE '2035-12-31', INTERVAL 1 DAY)) AS date_actual
),

dated_cols As(
  SELECT
  -- Primary Key: Integer surrogate key (e.g., 20260819)
  CAST(date_format(date_actual, 'yyyyMMdd') AS INT) AS date_key,
  date_actual,
  
  -- Calendar Year & Quarter
  year(date_actual) AS year,
  quarter(date_actual) AS quarter,
  concat('Q', quarter(date_actual)) AS quarter_name,
  
  -- Month Attributes
  month(date_actual) AS month,
  date_format(date_actual, 'MMMM') AS month_name,
  date_format(date_actual, 'MMM') AS month_short,
  
  -- Day & Week Attributes
  dayofmonth(date_actual) AS day_of_month,
  dayofweek(date_actual) AS day_of_week,
  date_format(date_actual, 'EEEE') AS day_name,
  dayofyear(date_actual) AS day_of_year,
  weekofyear(date_actual) AS week_of_year,
  
  -- Flags
  CASE WHEN dayofweek(date_actual) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  
  -- South African Municipal Financial Year (July 1 to June 30)
  CASE 
    WHEN month(date_actual) >= 7 THEN year(date_actual) + 1 
    ELSE year(date_actual) 
  END AS fin_year,
  
  CASE 
    WHEN month(date_actual) IN (7, 8, 9) THEN 'Q1'
    WHEN month(date_actual) IN (10, 11, 12) THEN 'Q2'
    WHEN month(date_actual) IN (1, 2, 3) THEN 'Q3'
    ELSE 'Q4'
  END AS fin_quarter
  FROM date_seq
)

SELECT * FROM dated_cols

UNION ALL
-- fallback aliases
SELECT
  -1                         AS date_key,
  CAST('1900-01-01' AS DATE) AS date_actual,
  -1                         AS year,
  -1                         AS quarter,
  'Unmapped'                 AS quarter_name,
  -1                         AS month,
  'Unmapped'                 AS month_name,
  'Unmapped'                 AS month_short,
  -1                         AS day_of_month,
  -1                         AS day_of_week,
  'Unmapped'                 AS day_name,
  -1                         AS day_of_year,
  -1                         AS week_of_year,
  FALSE                      AS is_weekend,
  -1                         AS fin_year,
  'Unmapped'                 AS fin_quarter;
